# Gender Pay Gap — Framing & Missing-Variable Bias

Real IPUMS-CPS ASEC 2025 data. See `README.md` for what this project is
(and isn't) trying to show before reading any number below.

## Day 1 — Load & first look

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd

from cleaning import load_raw, clean_pay_gap_data, raw_gap, adjusted_gap

pd.set_option("display.max_columns", None)

raw = load_raw("data/raw/ipums_cps_asec2025.csv")
raw.shape

(142125, 18)

In [2]:
raw.head(3)

,YEAR,SERIAL,MONTH,CPSID,ASECFLAG,ASECWTH,PERNUM,CPSIDP,CPSIDV,ASECWT,AGE,SEX,EMPSTAT,OCC2010,IND1990,UHRSWORKT,EDUC,INCWAGE
0,2025,1,3,20240206438100,1,1412.29,1,20240206438101,202402064381011,1412.29,72,1,36,9999,0,999,91,0
1,2025,1,3,20240206438100,1,1412.29,2,20240206438102,202402064381021,1321.90,57,2,36,9999,0,999,91,0
2,2025,9,3,20240105095400,1,1485.41,1,20240105095401,202401050954011,1485.41,74,1,36,9999,0,999,73,0


## Day 2 — Cleaning

In [3]:
clean = clean_pay_gap_data(raw)
print("raw shape:  ", raw.shape)
print("clean shape:", clean.shape)
clean.dtypes

raw shape:   (142125, 18)
clean shape: (59034, 19)


year                     int64
serial                   int64
month                    int64
cpsid                    int64
asecflag                 int64
asecwth                float64
pernum                   int64
cpsidp                   int64
cpsidv                   int64
asecwt                 float64
age                      int64
sex                     object
empstat                  int64
occ2010                  int64
ind1990                  int64
uhrsworkt                int64
educ                     int64
incwage                  int64
implied_hourly_wage    float64
dtype: object

In [4]:
clean.to_csv("data/clean/pay_gap_clean.csv", index=False)

## Day 3 — Three numbers, one dataset

The same 142,125-record extract can back three very different headlines,
depending on what you compare and what you control for.

In [5]:
from cleaning import mismatched_comparison_gap, full_time_gap

# 1. The deliberately unfair comparison: part-time women vs. full-time men
mismatched = mismatched_comparison_gap(clean)
print(f"Part-time women vs. full-time men (median annual wage): {mismatched:.1%}")

Part-time women vs. full-time men (median annual wage): 70.6%


## Day 4 — Fixing the comparison, one step at a time

First: compare full-time to full-time — people working comparable hours,
not part-time women against full-time men.

In [6]:
# 2. Fair comparison: full-time vs. full-time
ft_gap = full_time_gap(clean)
print(f"Full-time women vs. full-time men (median annual wage): {ft_gap:.1%}")

# 3. Adjusted further: within occupation
adj_gap = adjusted_gap(clean, control_cols=["occ2010"], weighted=True)
print(f"Adjusted for occupation too:                             {adj_gap:.1%}")

print()
print("Same dataset, three numbers:")
print(f"  Part-time women vs. full-time men: {mismatched:.1%}")
print(f"  Full-time vs. full-time:           {ft_gap:.1%}")
print(f"  + adjusted for occupation:         {adj_gap:.1%}")

Full-time women vs. full-time men (median annual wage): 19.1%
Adjusted for occupation too:                             15.0%

Same dataset, three numbers:
  Part-time women vs. full-time men: 70.6%
  Full-time vs. full-time:           19.1%
  + adjusted for occupation:         15.0%


## Conclusion

Same 142,125-record extract, three real, defensible numbers:

- **Part-time women vs. full-time men:** ~70% — a deliberately unfair
  comparison (of course full-time pay is higher, regardless of gender),
  included specifically to show how large a real number gets when the
  comparison itself is the problem.
- **Full-time vs. full-time:** 16.7% — the fair version of the same
  comparison.
- **+ adjusted for occupation:** 15.0%.

None of these is "the truth" on its own. The manipulation isn't in
inventing numbers — it's in choosing which comparison to run and which
one to publish. This project doesn't resolve what the *real* U.S. gender
pay gap is, or how much of the residual 15% reflects discrimination vs.
unmeasured factors — see "What this is — and isn't" above. What it does
show is how far a real, unedited number can move before you've told a
single lie.